# 02 — Local BF16 Text2SQL baseline with vLLM

This notebook loads the exact pinned BF16 customization checkpoint that Notebook 03 will tune and freezes its score on all 100 Mini-Dev rows. It uses vLLM because NVIDIA's official Nemotron 3.5 Lightning Text2SQL runbook specifies vLLM for this architecture; the model's bundled `transformers.generate()` path is not the supported baseline.

**Required runtime:** use the NeMo container Jupyter opened by `launchable/setup.sh` through the Secure Link on host port **8889**. The host `.venv` is API-only and intentionally has no PyTorch.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
ARTIFACTS_DIR = Path(os.environ.get('NEMOTRON_ARTIFACTS_DIR', ROOT / 'artifacts')).expanduser().resolve()
DATA_DIR = ARTIFACTS_DIR / 'data/bird-text2sql'
BASELINE_PATH = ARTIFACTS_DIR / 'evaluation/baseline_local_bf16_text2sql.json'
print('Repository:', ROOT)
print('Artifacts:', ARTIFACTS_DIR)
print('Python:', sys.executable)


## 1. Verify the container, GPU, model identity, and frozen holdout


In [ ]:
subprocess.run([sys.executable, 'scripts/preflight.py', '--profile', 'inference'], check=True)
subprocess.run([
    sys.executable, 'scripts/prepare_text2sql.py',
    '--output-dir', str(DATA_DIR), '--evaluation-only',
], check=True)

import torch
from nemotron_ft_lab.constants import MODEL_ID, MODEL_REVISION
from nemotron_ft_lab.data import read_jsonl

eval_rows = read_jsonl(DATA_DIR / 'evaluation.jsonl')
manifest = json.loads((DATA_DIR / 'evaluation_manifest.json').read_text())
print('Model:', MODEL_ID)
print('Pinned revision:', MODEL_REVISION)
print('GPU(s):', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Evaluation rows:', len(eval_rows), manifest['difficulty_distribution'])


## 2. Run the local baseline in an isolated vLLM process

Process isolation matters: when evaluation finishes, vLLM exits and releases GPU memory before LoRA training. One 80 GB GPU is the conservative default; set `NEMOTRON_INFERENCE_GPUS=2` only if this model/backend combination has been rehearsed on the node.


In [ ]:
INFERENCE_GPUS = int(os.environ.get('NEMOTRON_INFERENCE_GPUS', '1'))
if not 1 <= INFERENCE_GPUS <= torch.cuda.device_count():
    raise RuntimeError(f'NEMOTRON_INFERENCE_GPUS must be between 1 and {torch.cuda.device_count()}.')
command = [
    sys.executable, 'scripts/evaluate_vllm.py',
    '--model', MODEL_ID, '--revision', MODEL_REVISION,
    '--data-dir', str(DATA_DIR), '--output', str(BASELINE_PATH),
    '--run-type', 'untuned-local-bf16-text2sql',
    '--tensor-parallel-size', str(INFERENCE_GPUS),
]
print('Launching:', ' '.join(command))
started = time.perf_counter()
subprocess.run(command, check=True)
print(f'Baseline process finished in {(time.perf_counter() - started) / 60:.1f} min')


In [ ]:
baseline = json.loads(BASELINE_PATH.read_text())
metrics = ('n', 'execution_accuracy', 'sql_valid_rate', 'sql_executable_rate', 'normalized_exact_match')
print(json.dumps({key: baseline[key] for key in metrics}, indent=2))
print()
print('Predictions:')
for row in baseline['rows'][:5]:
    print()
    print('Q:', row['question'])
    print('Gold:', row['expected_sql'])
    print('Generated:', row['generated'])
    print('Execution correct:', row['execution_correct'])


## 3. Optional context: compare available cloud reports on shared IDs


In [ ]:
from nemotron_ft_lab.evaluation import paired_execution_comparison

baseline_by_id = {row['example_id']: row for row in baseline['rows']}
for path in sorted((ARTIFACTS_DIR / 'evaluation').glob('baseline_api_*_text2sql_*.json')):
    cloud = json.loads(path.read_text())
    shared = [row['example_id'] for row in cloud['rows']]
    if not set(shared).issubset(baseline_by_id):
        continue
    local_shared = {'rows': [baseline_by_id[item] for item in shared]}
    result = paired_execution_comparison(cloud, local_shared)
    print(path.name, json.dumps({
        'cloud_accuracy': cloud['execution_accuracy'],
        'local_bf16_accuracy': sum(r['execution_correct'] for r in local_shared['rows']) / len(shared),
        'local_minus_cloud': result['absolute_execution_accuracy_gain'],
    }, indent=2))


## Result contract

Notebook 03 must reuse `baseline_local_bf16_text2sql.json` and the exact IDs in this report. A lower training loss or more parseable SQL is useful diagnostics, but the fine-tuning claim requires higher execution accuracy on the identical holdout.
